# Embeddings Deep Dive — How Semantic Search Works

This notebook shows what happens inside semantic search:
- How text becomes a 384-dimensional vector
- Why cosine similarity captures meaning
- Why 'vendors' matches 'suppliers' but BM25 can't

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Output dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Embed some words and see how close they are
words = ["supplier", "vendor", "warehouse", "hospital", "medication"]
embeddings = model.encode(words)

print(f"Each word → {embeddings.shape[1]}-dimensional vector")
print(f"\nFirst 10 dimensions of 'supplier': {embeddings[0][:10].round(4)}")
print(f"First 10 dimensions of 'vendor':   {embeddings[1][:10].round(4)}")
print("\nThey look similar because they MEAN similar things!")

## Cosine Similarity Matrix

Cosine similarity measures the angle between two vectors.
- 1.0 = identical direction (same meaning)
- 0.0 = perpendicular (unrelated)
- -1.0 = opposite (rare for text)

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Build similarity matrix
print(f"{'':>12}", end="")
for w in words:
    print(f"{w:>12}", end="")
print()

for i, w1 in enumerate(words):
    print(f"{w1:>12}", end="")
    for j, w2 in enumerate(words):
        sim = cosine_sim(embeddings[i], embeddings[j])
        print(f"{sim:>12.4f}", end="")
    print()

In [ ]:
# The key demo: synonyms that BM25 misses
pairs = [
    ("Find vendors in China", "Shanghai Electronics Co. is a supplier based in Shanghai, China"),
    ("high blood sugar", "Hyperglycemia management protocol"),
    ("treatment options", "Type 2 diabetes treatment algorithm"),
]

print("Semantic similarity for query-document pairs:\n")
for query, doc in pairs:
    q_emb = model.encode(query)
    d_emb = model.encode(doc)
    sim = cosine_sim(q_emb, d_emb)
    print(f"  Query: \"{query}\"")
    print(f"  Doc:   \"{doc}\"")
    print(f"  Similarity: {sim:.4f}")
    print()

## Key Takeaways

1. **Embeddings capture meaning** — 'vendor' and 'supplier' are close in vector space
2. **384 dimensions** are enough to encode semantic relationships for most tasks
3. **Cosine similarity** works because direction matters more than magnitude
4. **The model learned this from millions of text pairs** — not from rules we wrote

Limitation: Semantic search can be LESS precise than BM25 for exact terms like 'metformin 500mg'.